# CT organ segmentation demo — TotalSegmentator

Runs `experts/ct_totalsegmentator.py`'s underlying model (TotalSegmentator, Apache
2.0, no gate, no license restriction — unlike MAIRA-2) against one real case from
the model's own official small-subset release (Zenodo record `10047263`, 102
subjects, ~3.2GB, v2.0.1). Not evaluated through the Gate 1-4 statistical protocol
used for KAD-512 — segmentation quality here is a different kind of claim, already
backed by TotalSegmentator's own published validation. This notebook proves the
pipeline actually runs end-to-end and prints real measured structure volumes; it
does **not** yet produce a DICOM SEG file or show anything inside the OHIF viewer
— that needs a NIfTI→DICOM synthesis step first (there's no DICOM series in this
dataset to attach a segmentation to), scoped as separate follow-up work.

What to do:
1. Connect to any available GPU runtime (an L4 has plenty of VRAM for the
   full-quality model — no need for TotalSegmentator's lower-resolution `fast`
   mode).
2. Select **Run all**. The ~3.2GB sample and the segmentation output are both
   cached under Google Drive → `MyDrive/doctor_assistant/totalsegmentator_demo`
   so re-running doesn't re-download or re-extract.


In [1]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/Hamza09Hamza/doctor_assistant.git"
REPO_REF = "classifier-evaluation-colab"
REPO_DIR = Path("/content/doctor_assistant")

if REPO_DIR.exists():
    result = subprocess.run(
        ["git", "status", "--porcelain"], cwd=REPO_DIR, capture_output=True, text=True
    )
    if result.stdout.strip():
        raise RuntimeError(
            f"{REPO_DIR} has local changes; remove it or commit/stash before re-cloning."
        )
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{REPO_REF}"], cwd=REPO_DIR, check=True)
else:
    subprocess.run(
        ["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True
    )

sys.path.insert(0, str(REPO_DIR))
print("Repo ready at", REPO_DIR)


Repo ready at /content/doctor_assistant


In [2]:
%pip install -q totalsegmentator


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 51.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 144.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.0/596.0 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6

In [3]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU is attached. Select any available GPU runtime, reconnect, then Run all.")
print("GPU:", torch.cuda.get_device_name(0))


GPU: NVIDIA L4


In [4]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_CACHE = Path("/content/drive/MyDrive/doctor_assistant/totalsegmentator_demo")
DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
print("Cache dir:", DRIVE_CACHE)


Mounted at /content/drive
Cache dir: /content/drive/MyDrive/doctor_assistant/totalsegmentator_demo


In [ ]:
# Omit --fast: on an L4 there's enough VRAM for the full-quality "big" model, which is
# what you actually want to evaluate rather than the lower-resolution fallback.
#
# PYTHONUNBUFFERED=1 matters here: Python fully buffers stdout (not line-by-line) when
# it isn't attached to a real terminal, which is exactly the case for a subprocess
# spawned from a notebook cell -- without this, every print() in demo_totalsegmentator.py
# gets held internally and you see nothing at all until the process exits, even though
# it's working the whole time (this bit us on the first real run: a 10+ minute silent
# cell during the Zenodo download/extract phase, before GPU work even starts).
import os

env = {**os.environ, "PYTHONUNBUFFERED": "1"}
result = subprocess.run(
    [
        sys.executable, "-u", "scripts/demo_totalsegmentator.py",
        "--drive-cache", str(DRIVE_CACHE),
        "--subject-index", "0",
    ],
    cwd=REPO_DIR,
    env=env,
)
if result.returncode != 0:
    raise RuntimeError(f"demo_totalsegmentator.py exited with code {result.returncode}")


## Next steps (not run here)

- Try a different `--subject-index` (0-101) to see segmentation on other cases.
- Drop `--fast` if you want to compare quality against the lower-resolution mode.
- The real remaining engineering work: synthesize a DICOM series from the NIfTI CT
  (so the segmentation has a real SOPInstanceUID series to attach a DICOM SEG to),
  export the mask as DICOM SEG, and load both into OHIF's `dicomlocal` data source
  (no Orthanc/server needed for a local demo) to actually see it in the Clinique
  Amina viewer.
